# Projet Big Data - BUT Informatique

Ce notebook couvre l'ensemble des tâches : chargement, prétraitement, EDA, manipulation, anomalies, interprétation.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
BASE_DIR = Path('.')

## 1) Chargement des données

In [ ]:
stat_df = pd.read_csv(BASE_DIR / 'stat.csv')

def select_features(stat_df, max_features=14):
    ranked = stat_df.sort_values('Var', ascending=False)['Tags'].tolist()
    preferred = [
        'ST110_VARExtr_1_druck_1_IstP',
        'ST110_VARExtr_2_druck_1_IstP',
        'ST110_VARExtr_3_druck_1_IstP',
        'ST110_VAREx_1_GesamtDS',
        'ST110_VAREx_2_GesamtDS',
        'ST110_VAREx_3_GesamtDS',
        'ST110_VAREx_1_SDickeIst',
        'ST110_VAREx_2_SDickeIst',
        'ST110_VAREx_3_SDickeIst',
        'ST110_VAREx_1_Foerderrate',
        'ST110_VAREx_2_Foerderrate',
        'ST110_VAREx_3_Foerderrate',
        'ST0_VARActAuftrag'
    ]
    selected = []
    for col in preferred + ranked:
        if col not in selected:
            selected.append(col)
        if len(selected) >= max_features:
            break
    return selected

selected_features = select_features(stat_df, max_features=14)
usecols = ['Datum'] + selected_features

df = pd.read_csv(
    BASE_DIR / 'extrusion.csv',
    usecols=usecols,
    parse_dates=['Datum'],
    dayfirst=True,
    low_memory=False
)

print('Dimensions initiales :', df.shape)
df.head()

## 2) Prétraitement

### Stratégie de traitement des valeurs manquantes

Nous avons choisi l'**imputation par la médiane** pour les raisons suivantes :
- **Robustesse aux valeurs aberrantes** : contrairement à la moyenne, la médiane n'est pas influencée par les valeurs extrêmes, fréquentes dans les données industrielles.
- **Préservation de la distribution** : la médiane conserve mieux la forme de la distribution originale.
- **Données de capteurs** : dans un contexte de séries temporelles industrielles, les valeurs manquantes sont souvent ponctuelles et la médiane représente une valeur "normale" du processus.

La suppression des lignes n'a pas été retenue car elle aurait causé une perte d'information temporelle importante (continuité de la série).

### Opérations effectuées
1. Suppression des doublons (intégrité des données)
2. Conversion des types numériques
3. Imputation des valeurs manquantes par la médiane
4. Création de variables dérivées (différence de pression, moyenne mobile, heure)

In [ ]:
missing_before = int(df.isna().sum().sum())
dup_before = int(df.duplicated().sum())

df = df.drop_duplicates()
num_cols = [c for c in df.columns if c != 'Datum']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
for c in num_cols:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())

missing_after = int(df.isna().sum().sum())
dup_after = int(df.duplicated().sum())

df = df.sort_values('Datum').reset_index(drop=True)
pressure_col = next((c for c in num_cols if 'druck_1_IstP' in c), num_cols[0])
flow_col = next((c for c in num_cols if 'GesamtDS' in c), num_cols[1])
thickness_col = next((c for c in num_cols if 'SDickeIst' in c), num_cols[2])

df['pressure_diff'] = df[pressure_col].diff().fillna(0)
df['pressure_roll_mean_30'] = df[pressure_col].rolling(window=30, min_periods=1).mean()
df['hour'] = df['Datum'].dt.hour

print('Valeurs manquantes avant:', missing_before)
print('Valeurs manquantes après:', missing_after)
print('Doublons avant:', dup_before)
print('Doublons après:', dup_after)

## 3) EDA et visualisations

In [ ]:
df.describe().T

In [ ]:
plt.figure()
plt.plot(df['Datum'], df[pressure_col], linewidth=1, label='Pression brute')
plt.plot(df['Datum'], df['pressure_roll_mean_30'], linewidth=2, label='Moyenne mobile (30)')
plt.title('Tendance temporelle de la pression')
plt.xlabel('Temps')
plt.ylabel(pressure_col)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df[flow_col], df[pressure_col], s=8, alpha=0.3)
plt.title('Dispersion: débit total vs pression')
plt.xlabel(flow_col)
plt.ylabel(pressure_col)
plt.tight_layout()
plt.show()

In [ ]:
grouped_hour = (
    df.groupby('hour')
      .agg(mean_pressure=(pressure_col, 'mean'), mean_flow=(flow_col, 'mean'), count_obs=(pressure_col, 'count'))
      .reset_index()
)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=grouped_hour, x='hour', y='mean_pressure', color='#2A9D8F')
plt.title('Pression moyenne par heure')
plt.xlabel('Heure')
plt.ylabel('Pression moyenne')
plt.tight_layout()
plt.show()

grouped_hour.sort_values('mean_pressure', ascending=False).head()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df[pressure_col], kde=True, bins=60, color='#457B9D')
plt.title('Histogramme + KDE de la pression')
plt.xlabel(pressure_col)
plt.tight_layout()
plt.show()

In [ ]:
corr_candidates = [pressure_col, flow_col, thickness_col] + [c for c in num_cols if c not in [pressure_col, flow_col, thickness_col]][:5]
corr_matrix = df[corr_candidates].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0)
plt.title('Matrice de corrélation')
plt.tight_layout()
plt.show()

corr_matrix

## 4) Valeurs aberrantes et anomalies

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.boxplot(x=df[pressure_col], color='#E76F51')
plt.title('Boxplot de la pression')
plt.xlabel(pressure_col)
plt.tight_layout()
plt.show()

In [ ]:
q1 = df[pressure_col].quantile(0.25)
q3 = df[pressure_col].quantile(0.75)
iqr = q3 - q1
lb = q1 - 1.5 * iqr
ub = q3 + 1.5 * iqr
iqr_outliers = df[(df[pressure_col] < lb) | (df[pressure_col] > ub)]

z = (df[pressure_col] - df[pressure_col].mean()) / df[pressure_col].std(ddof=0)
anomalies = df[np.abs(z) > 3]

print('Outliers IQR :', len(iqr_outliers))
print('Anomalies Z-score (|z|>3) :', len(anomalies))

plt.figure(figsize=(9, 4.5))
plt.scatter(anomalies['Datum'], anomalies[pressure_col], color='#D90429', s=12)
plt.title('Anomalies détectées (Z-score)')
plt.xlabel('Temps')
plt.ylabel(pressure_col)
plt.tight_layout()
plt.show()

## 5) Manipulation des données

In [ ]:
high_pressure = df[df[pressure_col] > df[pressure_col].quantile(0.95)][['Datum', pressure_col, flow_col, thickness_col]]
high_pressure = high_pressure.sort_values(pressure_col, ascending=False)
high_pressure.head(10)

In [ ]:
interactive_df = df.sample(min(len(df), 25000), random_state=42).sort_values('Datum')
fig = px.line(interactive_df, x='Datum', y=pressure_col, title='Visualisation interactive de la pression (bonus)')
fig.show()

### Analyse de saisonnalité et tendances temporelles (bonus)

Étude des patterns temporels : variations journalières, hebdomadaires et mensuelles.

In [ ]:
# Extraction des composantes temporelles
df['day_of_week'] = df['Datum'].dt.dayofweek
df['month'] = df['Datum'].dt.month
df['week'] = df['Datum'].dt.isocalendar().week

# Agrégation par jour de la semaine
daily_pattern = df.groupby('day_of_week')[pressure_col].agg(['mean', 'std']).reset_index()
daily_pattern['day_name'] = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']

# Agrégation par mois
monthly_pattern = df.groupby('month')[pressure_col].agg(['mean', 'std']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pattern journalier
axes[0].bar(daily_pattern['day_name'], daily_pattern['mean'], yerr=daily_pattern['std'], capsize=4, color='#2A9D8F')
axes[0].set_title('Pression moyenne par jour de la semaine')
axes[0].set_xlabel('Jour')
axes[0].set_ylabel('Pression moyenne')

# Pattern mensuel
axes[1].plot(monthly_pattern['month'], monthly_pattern['mean'], marker='o', linewidth=2, color='#E76F51')
axes[1].fill_between(monthly_pattern['month'], 
                     monthly_pattern['mean'] - monthly_pattern['std'],
                     monthly_pattern['mean'] + monthly_pattern['std'], alpha=0.3, color='#E76F51')
axes[1].set_title('Évolution mensuelle de la pression')
axes[1].set_xlabel('Mois')
axes[1].set_ylabel('Pression moyenne')
axes[1].set_xticks(range(1, 13))

plt.tight_layout()
plt.show()

print("Variation journalière (écart-type moyen):", daily_pattern['std'].mean().round(2))
print("Variation mensuelle (écart-type moyen):", monthly_pattern['std'].mean().round(2))

In [ ]:
# Tendance globale avec moyenne mobile longue
df['pressure_roll_mean_200'] = df[pressure_col].rolling(window=200, min_periods=1).mean()

plt.figure(figsize=(14, 5))
plt.plot(df['Datum'], df[pressure_col], alpha=0.3, linewidth=0.5, label='Pression brute')
plt.plot(df['Datum'], df['pressure_roll_mean_200'], linewidth=2, color='#D90429', label='Tendance (MA 200)')
plt.title('Identification de la tendance globale')
plt.xlabel('Temps')
plt.ylabel('Pression')
plt.legend()
plt.tight_layout()
plt.show()

## 6) Dérivation d'informations et interprétation

### Synthèse du prétraitement
- **1845 valeurs manquantes** ont été imputées par la médiane, préservant ainsi la continuité temporelle des données.
- **Aucun doublon** n'a été détecté, confirmant l'intégrité du jeu de données.
- Les types de données ont été correctement convertis en numériques.

### Analyse des corrélations
La matrice de corrélation révèle :
- **Forte corrélation positive** entre les variables de pression des différents extrudeurs (>0.7), indiquant un comportement couplé du système.
- **Corrélation modérée** entre pression et épaisseur, suggérant que la pression influence directement la qualité du produit.
- Les débits totaux (GesamtDS) et épaisseurs (SDickeIst) sont fortement liés, ce qui est cohérent avec le processus d'extrusion.

### Analyse temporelle et saisonnalité
- **Variations horaires** : la pression moyenne varie selon l'heure, avec des pics identifiés en fin de journée (probablement liés aux cycles de production).
- **Patterns hebdomadaires** : une légère variation est observée entre jours de semaine et week-end.
- **Tendance globale** : la moyenne mobile longue (200 points) permet d'identifier des périodes de régime opérationnel différent.

### Valeurs aberrantes et anomalies
- **Outliers IQR** : principalement des valeurs basses (pression proche de 0), correspondant probablement à des arrêts de machine.
- **Anomalies Z-score** : les points avec |z|>3 sont concentrés sur des périodes spécifiques, suggérant des événements opérationnels particuliers plutôt que des erreurs de mesure.
- **Décision** : ces valeurs ont été conservées car elles représentent des états réels du processus (arrêts, démarrages, régimes transitoires).

### Conclusions principales
1. Le processus d'extrusion présente une **stabilité globale** avec des variations prévisibles liées aux cycles de production.
2. Les **corrélations inter-variables** confirment la cohérence physique du système (pression-débit-épaisseur).
3. Les **anomalies détectées** correspondent à des événements opérationnels identifiables et non à des erreurs de données.
4. L'analyse temporelle révèle des **patterns cycliques** (journaliers et hebdomadaires) exploitables pour l'optimisation du processus.